In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
tables = ["account", "client", "disp", "order", "trans", "loan", "card", "district"]

def detect_sep(path):
    with open(path, encoding="utf-8", errors="ignore") as f:
        first_line = f.readline()
    return ";" if first_line.count(";") > first_line.count(",") else ","

dfs = {}
for t in tables:
    path = RAW / f"{t}.csv"
    sep = detect_sep(path)
    dfs[t] = pd.read_csv(path, sep=sep, low_memory=False)
    print(f"✅ {t:10s} sep='{sep}'  {dfs[t].shape}")

✅ account    sep=';'  (4500, 4)
✅ client     sep=';'  (5369, 3)
✅ disp       sep=';'  (5369, 4)
✅ order      sep=';'  (6471, 6)
✅ trans      sep=';'  (1056320, 10)
✅ loan       sep=';'  (682, 7)
✅ card       sep=';'  (892, 4)
✅ district   sep=';'  (77, 16)


In [7]:
account  = dfs["account"]
client   = dfs["client"]
disp     = dfs["disp"]
order    = dfs["order"]
trans    = dfs["trans"]
loan     = dfs["loan"]
card     = dfs["card"]
district = dfs["district"]

In [9]:
account.head()       

,account_id,district_id,frequency,date
0,576,55,POPLATEK MESICNE,930101
1,3818,74,POPLATEK MESICNE,930101
2,704,55,POPLATEK MESICNE,930101
3,2378,16,POPLATEK MESICNE,930101
4,2632,24,POPLATEK MESICNE,930102


In [10]:
account.shape         

(4500, 4)

In [ ]:
account.info()       

<class 'pandas.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   account_id   4500 non-null   int64
 1   district_id  4500 non-null   int64
 2   frequency    4500 non-null   str  
 3   date         4500 non-null   int64
dtypes: int64(3), str(1)
memory usage: 140.8 KB


In [12]:
account.describe()    # statistiques des colonnes numériques

,account_id,district_id,date
count,4500.000000,4500.000000,4500.000000
mean,2786.067556,37.310444,951654.608667
std,2313.811984,25.177217,14842.188377
min,1.000000,1.000000,930101.000000
25%,1182.750000,13.000000,931227.000000
50%,2368.000000,38.000000,960102.000000
75%,3552.250000,60.000000,961101.000000
max,11382.000000,77.000000,971229.000000


In [13]:
account.describe(include="object")   # statistiques des colonnes texte

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14228\2972964764.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  account.describe(include="object")   # statistiques des colonnes texte


,frequency
count,4500
unique,3
top,POPLATEK MESICNE
freq,4167


In [14]:
account.isna().sum()  # valeurs manquantes par colonne

account_id     0
district_id    0
frequency      0
date           0
dtype: int64

In [15]:
account["frequency"].value_counts()

frequency
POPLATEK MESICNE      4167
POPLATEK TYDNE         240
POPLATEK PO OBRATU      93
Name: count, dtype: int64

In [16]:
print("account_id unique :", account["account_id"].is_unique)
print("districts inconnus :", (~account["district_id"].isin(district.iloc[:, 0])).sum())

account_id unique : True
districts inconnus : 0


- `account_id` unique → clé primaire OK
- Intégrité référentielle `district_id` → district : OK (0 orphelin)

In [17]:
client.head()

,client_id,birth_number,district_id
0,1,706213,18
1,2,450204,1
2,3,406009,1
3,4,561201,5
4,5,605703,5


In [18]:
client.info()


<class 'pandas.DataFrame'>
RangeIndex: 5369 entries, 0 to 5368
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   client_id     5369 non-null   int64
 1   birth_number  5369 non-null   int64
 2   district_id   5369 non-null   int64
dtypes: int64(3)
memory usage: 126.0 KB


In [19]:
bn = client["birth_number"]
mois = (bn // 100) % 100
print("Mois min / max :", mois.min(), "/", mois.max())
print("Clients avec mois > 12 :", (mois > 12).sum())

Mois min / max : 1 / 62
Clients avec mois > 12 : 2645


### Observations — client
- 5 369 clients, aucune valeur manquante
- `birth_number` encode date de naissance ET sexe : YYMMDD, mois + 50 pour les femmes
- 2 645 femmes / 2 724 hommes → Silver : créer `gender` + `birth_date`, supprimer `birth_number`

In [20]:
disp.head()

,disp_id,client_id,account_id,type
0,1,1,1,OWNER
1,2,2,2,OWNER
2,3,3,2,DISPONENT
3,4,4,3,OWNER
4,5,5,3,DISPONENT


In [21]:
disp.info()


<class 'pandas.DataFrame'>
RangeIndex: 5369 entries, 0 to 5368
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   disp_id     5369 non-null   int64
 1   client_id   5369 non-null   int64
 2   account_id  5369 non-null   int64
 3   type        5369 non-null   str  
dtypes: int64(3), str(1)
memory usage: 167.9 KB


In [22]:
disp["type"].value_counts()

type
OWNER        4500
DISPONENT     869
Name: count, dtype: int64

In [ ]:
print("Clients par compte :")
print(disp.groupby("account_id").size().value_counts().sort_index())

owners = disp[disp["type"] == "OWNER"].groupby("account_id").size()
print("\nComptes avec 1 seul OWNER :", (owners == 1).sum(), "/", account.shape[0])

print("\nComptes par client :")
print(disp.groupby("client_id").size().value_counts().sort_index())

print("\nclients inconnus :", (~disp["client_id"].isin(client["client_id"])).sum())
print("comptes inconnus :", (~disp["account_id"].isin(account["account_id"])).sum())

Clients par compte :
1    3631
2     869
Name: count, dtype: int64

Comptes avec 1 seul OWNER : 4500 / 4500

Comptes par client :
1    5369
Name: count, dtype: int64

clients inconnus : 0
comptes inconnus : 0


### Observations — disp
- 5 369 lignes : 4 500 OWNER + 869 DISPONENT
- Chaque compte a exactement 1 OWNER ; 869 comptes joints (2 personnes)
- Chaque client est lié à 1 seul compte → relation simple
- Intégrité référentielle OK (0 orphelin)
- Gold : ajouter `owner_client_id` et `is_joint_account` dans dim_account
- ⚠️ `disp_id` sert de clé à la table `card`

In [24]:
order.head()

,order_id,account_id,bank_to,account_to,amount,k_symbol
0,29401,1,YZ,87144583,2452.0,SIPO
1,29402,2,ST,89597016,3372.7,UVER
2,29403,2,QR,13943797,7266.0,SIPO
3,29404,3,WX,83084338,1135.0,SIPO
4,29405,3,CD,24485939,327.0,


In [25]:
order.info()

<class 'pandas.DataFrame'>
RangeIndex: 6471 entries, 0 to 6470
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    6471 non-null   int64  
 1   account_id  6471 non-null   int64  
 2   bank_to     6471 non-null   str    
 3   account_to  6471 non-null   int64  
 4   amount      6471 non-null   float64
 5   k_symbol    6471 non-null   str    
dtypes: float64(1), int64(3), str(2)
memory usage: 303.5 KB


In [26]:
order["k_symbol"].value_counts(dropna=False)

k_symbol
SIPO        3502
            1379
UVER         717
POJISTNE     532
LEASING      341
Name: count, dtype: int64

In [27]:
order.describe()

,order_id,account_id,account_to,amount
count,6471.000000,6471.000000,6.471000e+03,6471.000000
mean,33778.197497,2962.302890,4.939904e+07,3280.635698
std,3737.681949,2518.503228,2.888356e+07,2714.475335
min,29401.000000,1.000000,3.990000e+02,1.000000
25%,31187.500000,1223.000000,2.415918e+07,1241.500000
50%,32988.000000,2433.000000,4.975606e+07,2596.000000
75%,34785.500000,3645.500000,7.400045e+07,4613.500000
max,46338.000000,11362.000000,9.999420e+07,14882.000000


In [28]:
print("order_id unique :", order["order_id"].is_unique)
print("comptes inconnus :", (~order["account_id"].isin(account["account_id"])).sum())
print("Valeurs vides ou espaces dans k_symbol :", order["k_symbol"].astype(str).str.strip().eq("").sum())

order_id unique : True
comptes inconnus : 0
Valeurs vides ou espaces dans k_symbol : 1379


In [ ]:
order.nsmallest(10, "amount")

,order_id,account_id,bank_to,account_to,amount,k_symbol
1498,31044,1125,YZ,64038717,1.0,POJISTNE
1822,31414,1366,UV,48775595,1.0,POJISTNE
3326,33082,2491,ST,7648384,1.0,SIPO
3839,33646,2865,OP,25167072,1.0,
308,29742,225,KL,93545496,2.0,
526,29970,378,ST,70710897,2.0,POJISTNE
2752,32423,2058,IJ,17472982,2.0,
3838,33645,2865,ST,96803217,2.0,POJISTNE
3910,33717,2916,WX,87459617,2.0,
5074,35022,3809,YZ,9162903,2.0,POJISTNE


In [ ]:
comptes_uver = set(order.loc[order["k_symbol"] == "UVER", "account_id"])
comptes_pret = set(loan["account_id"])
print("Comptes avec ordre UVER :", len(comptes_uver))
print("Comptes avec un prêt    :", len(comptes_pret))
print("UVER sans prêt          :", len(comptes_uver - comptes_pret))

Comptes avec ordre UVER : 717
Comptes avec un prêt    : 682
UVER sans prêt          : 35


- Montants 1–2 (surtout POJISTNE / vides) : conservés, flag `is_suspicious_amount` en Silver
- Cohérence order ↔ loan : les 682 prêts ont tous un ordre UVER ✅ ; 35 ordres UVER sans prêt → probablement prêts dans une autre banque

In [31]:
loan.head()

,loan_id,account_id,date,amount,duration,payments,status
0,5314,1787,930705,96396,12,8033.0,B
1,5316,1801,930711,165960,36,4610.0,A
2,6863,9188,930728,127080,60,2118.0,A
3,5325,1843,930803,105804,36,2939.0,A
4,7240,11013,930906,274740,60,4579.0,A


In [32]:
loan.info()

<class 'pandas.DataFrame'>
RangeIndex: 682 entries, 0 to 681
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   loan_id     682 non-null    int64  
 1   account_id  682 non-null    int64  
 2   date        682 non-null    int64  
 3   amount      682 non-null    int64  
 4   duration    682 non-null    int64  
 5   payments    682 non-null    float64
 6   status      682 non-null    str    
dtypes: float64(1), int64(5), str(1)
memory usage: 37.4 KB


In [33]:
loan["status"].value_counts().sort_index()

status
A    203
B     31
C    403
D     45
Name: count, dtype: int64

In [34]:
loan["duration"].value_counts().sort_index()

duration
12    131
24    138
36    130
48    138
60    145
Name: count, dtype: int64

In [35]:
loan[["amount", "payments"]].describe()

,amount,payments
count,682.000000,682.000000
mean,151410.175953,4190.664223
std,113372.406310,2215.830344
min,4980.000000,304.000000
25%,66732.000000,2477.000000
50%,116928.000000,3934.000000
75%,210654.000000,5813.500000
max,590820.000000,9910.000000


In [36]:
print("loan_id unique :", loan["loan_id"].is_unique)
print("Prêts par compte :", loan.groupby("account_id").size().value_counts().to_dict())
print("comptes inconnus :", (~loan["account_id"].isin(account["account_id"])).sum())

# Cohérence : montant ≈ durée × mensualité ?
ecart = (loan["amount"] - loan["duration"] * loan["payments"]).abs()
print("Prêts où montant ≠ durée × mensualité :", (ecart > 1).sum())

# Le prêt est-il accordé APRÈS l'ouverture du compte ?
m = loan.merge(account[["account_id", "date"]], on="account_id", suffixes=("_loan", "_account"))
d_loan = pd.to_datetime(m["date_loan"].astype(str), format="%y%m%d")
d_acc  = pd.to_datetime(m["date_account"].astype(str), format="%y%m%d")
print("Prêts accordés avant l'ouverture du compte :", (d_loan < d_acc).sum())
print("Période des prêts :", d_loan.min().date(), "→", d_loan.max().date())

loan_id unique : True
Prêts par compte : {1: 682}
comptes inconnus : 0
Prêts où montant ≠ durée × mensualité : 0
Prêts accordés avant l'ouverture du compte : 0
Période des prêts : 1993-07-05 → 1998-12-08


### Observations — loan
- 682 prêts, table très propre : clés OK, 0 orphelin, 1 prêt max par compte
- `amount` = `duration` × `payments` (vérifié sur 100 % des prêts)
- `status` : A=terminé OK, B=terminé impayé, C=en cours OK, D=en cours en défaut
- Taux de défaut (B+D) : 11,1 % → KPI principal de la page risque
- Silver : créer `loan_state` (FINISHED/RUNNING) + `is_default` (bool), convertir `date`
- Période des prêts : 1993-07 → 1998-12 → dim_date doit couvrir 1993–1998

In [37]:
card.head()

,card_id,disp_id,type,issued
0,1005,9285,classic,931107 00:00:00
1,104,588,classic,940119 00:00:00
2,747,4915,classic,940205 00:00:00
3,70,439,classic,940208 00:00:00
4,577,3687,classic,940215 00:00:00


In [38]:
card.info()

<class 'pandas.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   card_id  892 non-null    int64
 1   disp_id  892 non-null    int64
 2   type     892 non-null    str  
 3   issued   892 non-null    str  
dtypes: int64(2), str(2)
memory usage: 28.0 KB


In [39]:
card["type"].value_counts()

type
classic    659
junior     145
gold        88
Name: count, dtype: int64

In [41]:
print("card_id unique :", card["card_id"].is_unique)
print("disp inconnus :", (~card["disp_id"].isin(disp["disp_id"])).sum())

c = card.merge(disp[["disp_id", "account_id", "type"]], on="disp_id", suffixes=("_card", "_disp"))
print("\nCartes par type de titulaire :")
print(c["type_disp"].value_counts())
print("\nCartes par compte :", c.groupby("account_id").size().value_counts().to_dict())

card_id unique : True
disp inconnus : 0

Cartes par type de titulaire :
type_disp
OWNER    892
Name: count, dtype: int64

Cartes par compte : {1: 892}


In [42]:
c = card.merge(disp[["disp_id", "account_id", "client_id"]], on="disp_id")
c["issued_date"] = pd.to_datetime(c["issued"].str[:6], format="%y%m%d")

# 1. La carte est-elle émise après l'ouverture du compte ?
c = c.merge(account[["account_id", "date"]], on="account_id")
c["account_date"] = pd.to_datetime(c["date"].astype(str), format="%y%m%d")
print("Cartes émises avant l'ouverture du compte :", (c["issued_date"] < c["account_date"]).sum())
print("Période d'émission :", c["issued_date"].min().date(), "→", c["issued_date"].max().date())

# 2. Les cartes "junior" sont-elles bien détenues par des jeunes ?
bn = client["birth_number"]
mois = (bn // 100) % 100
client_bd = pd.DataFrame({
    "client_id": client["client_id"],
    "birth_date": pd.to_datetime(dict(year=1900 + bn // 10000,
                                      month=mois.where(mois <= 50, mois - 50),
                                      day=bn % 100))
})
c = c.merge(client_bd, on="client_id")
c["age_at_issue"] = (c["issued_date"] - c["birth_date"]).dt.days // 365
print("\nÂge du titulaire à l'émission, par type de carte :")
print(c.groupby("type")["age_at_issue"].describe()[["min", "mean", "max"]].round(1))

Cartes émises avant l'ouverture du compte : 0
Période d'émission : 1993-11-07 → 1998-12-29

Âge du titulaire à l'émission, par type de carte :
          min  mean   max
type                     
classic  20.0  41.6  80.0
gold     20.0  42.6  68.0
junior   12.0  17.7  20.0


- 0 carte émise avant l'ouverture du compte ✅ ; période : 1993-11 → 1998-12
- Règle métier vérifiée : junior = 12–20 ans, classic/gold ≥ 20 ans → décodage birth_number validé
- Clients mineurs titulaires de compte (dès 12 ans) → segment d'âge utile pour le dashboard

In [43]:
district.head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A16
0,1,Hl.m. Praha,Prague,1204953,0,0,0,1,1,100.0,12541,0.29,0.43,167,85677,99107
1,2,Benesov,central Bohemia,88884,80,26,6,2,5,46.7,8507,1.67,1.85,132,2159,2674
2,3,Beroun,central Bohemia,75232,55,26,4,1,5,41.7,8980,1.95,2.21,111,2824,2813
3,4,Kladno,central Bohemia,149893,63,29,6,2,6,67.4,9753,4.64,5.05,109,5244,5892
4,5,Kolin,central Bohemia,95616,65,30,4,1,6,51.4,9307,3.85,4.43,118,2616,3040


In [44]:
district.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A1      77 non-null     int64  
 1   A2      77 non-null     str    
 2   A3      77 non-null     str    
 3   A4      77 non-null     int64  
 4   A5      77 non-null     int64  
 5   A6      77 non-null     int64  
 6   A7      77 non-null     int64  
 7   A8      77 non-null     int64  
 8   A9      77 non-null     int64  
 9   A10     77 non-null     float64
 10  A11     77 non-null     int64  
 11  A12     77 non-null     str    
 12  A13     77 non-null     float64
 13  A14     77 non-null     int64  
 14  A15     77 non-null     str    
 15  A16     77 non-null     int64  
dtypes: float64(2), int64(10), str(4)
memory usage: 9.8 KB


In [45]:
for col in district.columns[3:]:
    if district[col].dtype == "str" or district[col].dtype == object:
        bizarres = district.loc[pd.to_numeric(district[col], errors="coerce").isna(), col]
        print(f"{col} : type texte → valeurs non numériques : {bizarres.unique().tolist()}")

A12 : type texte → valeurs non numériques : ['?']
A15 : type texte → valeurs non numériques : ['?']


In [46]:
print("district_id unique :", district.iloc[:, 0].is_unique)
print("Régions :", district.iloc[:, 2].value_counts().to_dict())

district_id unique : True
Régions : {'south Moravia': 14, 'central Bohemia': 12, 'east Bohemia': 11, 'north Moravia': 11, 'west Bohemia': 10, 'north Bohemia': 10, 'south Bohemia': 8, 'Prague': 1}


### Observations — district
- 77 districts, 8 régions, district_id unique
- Colonnes nommées A1–A16 → Silver : renommer (district_name, region, inhabitants, avg_salary…)
- ⚠️ A12 (chômage 95) et A15 (crimes 95) contiennent '?' → lues en texte. 1 district concerné
  → Silver : cast en numérique, '?' → NULL (ou imputation valeur 1996, à documenter)
- Prague = outlier (1,2 M hab.) → Gold : utiliser des taux pour 1 000 hab., pas des valeurs brutes

In [47]:
trans.head()

,trans_id,account_id,date,type,operation,amount,balance,k_symbol,bank,account
0,695247,2378,930101,PRIJEM,VKLAD,700.0,700.0,NaN,NaN,NaN
1,171812,576,930101,PRIJEM,VKLAD,900.0,900.0,NaN,NaN,NaN
2,207264,704,930101,PRIJEM,VKLAD,1000.0,1000.0,NaN,NaN,NaN
3,1117247,3818,930101,PRIJEM,VKLAD,600.0,600.0,NaN,NaN,NaN
4,579373,1972,930102,PRIJEM,VKLAD,400.0,400.0,NaN,NaN,NaN


In [48]:
trans.info()

<class 'pandas.DataFrame'>
RangeIndex: 1056320 entries, 0 to 1056319
Data columns (total 10 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   trans_id    1056320 non-null  int64  
 1   account_id  1056320 non-null  int64  
 2   date        1056320 non-null  int64  
 3   type        1056320 non-null  str    
 4   operation   873206 non-null   str    
 5   amount      1056320 non-null  float64
 6   balance     1056320 non-null  float64
 7   k_symbol    574439 non-null   str    
 8   bank        273508 non-null   str    
 9   account     295389 non-null   float64
dtypes: float64(3), int64(3), str(4)
memory usage: 80.6 MB


In [49]:
trans.isna().sum()

trans_id           0
account_id         0
date               0
type               0
operation     183114
amount             0
balance            0
k_symbol      481881
bank          782812
account       760931
dtype: int64

In [50]:
for col in ["type", "operation", "k_symbol"]:
    print(f"\n--- {col} ---")
    print(trans[col].value_counts(dropna=False))


--- type ---
type
VYDAJ     634571
PRIJEM    405083
VYBER      16666
Name: count, dtype: int64

--- operation ---
operation
VYBER             434918
PREVOD NA UCET    208283
NaN               183114
VKLAD             156743
PREVOD Z UCTU      65226
VYBER KARTOU        8036
Name: count, dtype: int64

--- k_symbol ---
k_symbol
NaN            481881
UROK           183114
SLUZBY         155832
SIPO           118065
                53433
DUCHOD          30338
POJISTNE        18500
UVER            13580
SANKC. UROK      1577
Name: count, dtype: int64


In [51]:
trans[["amount", "balance"]].describe()

,amount,balance
count,1.056320e+06,1.056320e+06
mean,5.924146e+03,3.851833e+04
std,9.522735e+03,2.211787e+04
min,0.000000e+00,-4.112570e+04
25%,1.359000e+02,2.240250e+04
50%,2.100000e+03,3.314340e+04
75%,6.800000e+03,4.960362e+04
max,8.740000e+04,2.096370e+05


In [52]:
print("trans_id unique :", trans["trans_id"].is_unique)
print("comptes inconnus :", (~trans["account_id"].isin(account["account_id"])).sum())
d = pd.to_datetime(trans["date"].astype(str), format="%y%m%d")
print("Période :", d.min().date(), "→", d.max().date())
print("Soldes négatifs :", (trans["balance"] < 0).sum())
print("Doublons complets :", trans.duplicated().sum())

trans_id unique : True
comptes inconnus : 0
Période : 1993-01-01 → 1998-12-31
Soldes négatifs : 2999
Doublons complets : 0


In [53]:
print(trans.loc[trans["type"] == "VYBER", "operation"].value_counts(dropna=False))

operation
VYBER    16666
Name: count, dtype: int64


In [54]:
print(pd.crosstab(trans["operation"].fillna("NaN"), trans["k_symbol"].fillna("NaN")).loc["NaN"])

k_symbol
                    0
DUCHOD              0
NaN                 0
POJISTNE            0
SANKC. UROK         0
SIPO                0
SLUZBY              0
UROK           183114
UVER                0
Name: NaN, dtype: int64


In [55]:
sank = trans[trans["k_symbol"] == "SANKC. UROK"]["account_id"].unique()
neg  = trans[trans["balance"] < 0]["account_id"].unique()
print("Comptes avec pénalités :", len(sank), "| dont déjà passés à découvert :", len(set(sank) & set(neg)))

Comptes avec pénalités : 264 | dont déjà passés à découvert : 255


In [56]:
print("Montants = 0 :", (trans["amount"] == 0).sum())
print("Compte partenaire sans banque :", (trans["account"].notna() & trans["bank"].isna()).sum())

Montants = 0 : 14
Compte partenaire sans banque : 21881


### Observations — trans (1 056 320 lignes)
- Clés OK : trans_id unique, 0 orphelin, 0 doublon ; période 1993-01-01 → 1998-12-31
- ⚠️ `type` : 3 valeurs (PRIJEM, VYDAJ + 16 666 « VYBER » = retrait mal classé) → Silver : VYBER → DEBIT
- `operation` NaN (183 114) = exactement les intérêts UROK → Silver : INTEREST_CREDIT
- ⚠️ `k_symbol` : 481 881 NaN + 53 433 espaces (~51 % vide) → Silver : trim + NULL
- `amount` toujours positif → Silver : créer `signed_amount` (négatif si débit)
- 2 999 soldes négatifs (découverts) liés aux pénalités SANKC. UROK
- `account` stocké en float → string ; ~22 000 lignes avec compte partenaire sans banque
- Traductions : VKLAD=dépôt, VYBER=retrait, PREVOD NA/Z UCTU=virement émis/reçu, DUCHOD=pension, SLUZBY=frais